# S5 manual check: progress, interrupt, re-run in JupyterLab

Run from `spikes/S5_chunks` with this venv's kernel: `uv run --with jupyterlab jupyter lab manual_check.ipynb`. Tick the checklist in `docs/design/spikes/S5.md` as you go.

In [ ]:
import sys

import s5lib

from brotkit.kernels.cache import default_root

gpu = s5lib.Gpu.open(default_root())
for tier in ("f32", "dd", "qf"):
    gpu.kernel(tier, "C")  # compiles once on a cold cache
print("ready", gpu.backend.target)

**1. Progress.** The line below should update in place about ten times a second and reach 100%. About 6 s (qf, worst-case view).

In [ ]:
r = s5lib.render(gpu, "qf", "C", "whole", on_chunk=s5lib.notebook_progress("qf whole 4K"))
print(r.complete, r.summary()["chunks"], "chunks, overhead", round(r.overhead * 100, 2), "%")

**2. Interrupt.** Run this, then press the stop button (or `I, I`) after a few seconds. It's about 80 s long (dd, all interior, max_iter 10000, chunks of about 0.3 s). Expect the cell to stop within the chunk in flight (about 0.3 s, never more than the 0.5 s ceiling) with `RenderInterrupted`, and the progress line to freeze below 100%. A second press stops at once, without recording the chunk in flight.

In [ ]:
s5lib.render(
    gpu,
    "dd",
    "C",
    "interior",
    sentinel=True,
    reraise=True,
    on_chunk=s5lib.notebook_progress("dd interior 4K"),
)

**3. Partial result.** Should print `complete False`, a done fraction below 1, and `True True` (finished tiles fully written, the rest untouched).

In [ ]:
R = sys.last_value.result
f, u = s5lib.host_layers(gpu, R)
cov = s5lib.coverage(R)
print(
    "complete",
    R.complete,
    "| done",
    round(float(cov.mean()), 3),
    "| done tiles written",
    bool((u[0][cov] != 0xFFFFFFFF).all()),
    "| rest untouched",
    bool((u[0][~cov] == 0xFFFFFFFF).all()),
)

**4. Re-run after the interrupt.** Should finish normally (complete True) with no recompile or reload (lookups still 3).

In [ ]:
r2 = s5lib.render(gpu, "f32", "C", "seahorse", on_chunk=s5lib.notebook_progress("f32 seahorse 4K"))
print("complete", r2.complete, "| kernel lookups", gpu.lookups)